In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from uni import get_encoder  # from the cloned UNI repo
import general_fcns as gf    # Your existing helper file
from huggingface_hub import login

# Setup
save_dir = './features_uni'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

login(token="")  # Insert your token

# Load UNI model and transform
model, transform = get_encoder(enc_name='uni2-h', device=device)
model.eval()

# Create output directory
os.makedirs(save_dir, exist_ok=True)

if torch.cuda.is_available():
    print("cuda")

# CSV path and skip list
csv_file_path = os.path.join(os.getcwd(), 'filtered_metadata_full_path.csv')
skip = ["D:/Digital_path_unzipped/SZMC1050237_pdl1.ndpi", "D:/Digital_path/DIG_PAT_1727366215.ndpi"]

# Read CSV and get image paths
df = pd.read_csv(csv_file_path)
image_paths = df.iloc[:, 0].dropna().tolist()  # Assuming paths are in the first column
image_paths = [p for p in image_paths if p not in skip]

valid_exts = ('.png', '.jpg', '.jpeg', '.tif', '.svs', '.ndpi')

# Process each image path from CSV
for img_path in image_paths:
    if not img_path.lower().endswith(valid_exts):
        continue

    img_file = os.path.basename(img_path)
    print(f"Processing: {img_file}")

    try:
        # Load image
        if img_path.lower().endswith('.svs') or img_path.lower().endswith('.ndpi') or img_path.lower().endswith('.tif'):
            slide = gf.slide_at_magnification(img_path, magnification_params={'magnification': 10})
            image = Image.fromarray(slide)
        else:
            image = Image.open(img_path).convert('RGB')

        # Preprocess image
        image = transform(image).unsqueeze(0).to(device)

        # Extract features
        with torch.inference_mode():
            feature_emb = model(image)

        # Save feature tensor
        save_path = os.path.join(save_dir, img_file.replace('.', '_') + '_feature.pt')
        torch.save(feature_emb.cpu(), save_path)
        print(f"Saved UNI feature to {save_path}")

    except Exception as e:
        print(f"Error processing {img_file}: {e}")


Processing: DIG_PAT_1701101606.tif


INFO:large_image:Cannot use memcached for caching.


Saved UNI feature to ./features_uni\DIG_PAT_1701101606_tif_feature.pt
Processing: DIG_PAT_1701103165.tif
Saved UNI feature to ./features_uni\DIG_PAT_1701103165_tif_feature.pt
Processing: DIG_PAT_1701103988.tif


KeyboardInterrupt: 